In [1]:
from io import StringIO
import re
import socket

import pandas as pd


# AE33 TCP/IP Communication Guide

This notebook demonstrates how to communicate with the **AE33 Aethalometer** using its TCP/IP interface for real-time data acquisition and control.


In [2]:
# IP of the instrument - change it to the actual IP address of your AE33 instrument
INSTRUMENT_IP = '10.10.10.224'

In [3]:
# default parameters:
INSTRUMENT_PORT = 8002  # always the same
BUFFER_SIZE = 4096  # always the same
RECV_TIMEOUT = 2.0  # seconds

COLUMNS_AE33_DATA = ['SerialNumber', 'ID', 'StartTime', 'EndTime', 'SetupID', 'SetupTimestamp', 'Ref1', 'Sens11', 'Sens12', 'Ref2', 'Sens21', 'Sens22', 'Ref3', 'Sens31', 'Sens32', 'Ref4', 'Sens41', 'Sens42', 'Ref5', 'Sens51', 'Sens52', 'Ref6', 'Sens61', 'Sens62', 'Ref7', 'Sens71', 'Sens72', 'BC11', 'BC12', 'BC1', 'BC21', 'BC22', 'BC2', 'BC31', 'BC32', 'BC3', 'BC41', 'BC42', 'BC4', 'BC51', 'BC52', 'BC5', 'BC61', 'BC62', 'BC6', 'BC71', 'BC72', 'BC7', 'K1', 'K2', 'K3', 'K4', 'K5', 'K6', 'K7', 'BB', 'Pressure', 'Temp', 'Flow1', 'Flow2', 'FlowC', 'T_controller', 'T_supply', 'T_LED', 'ControllerStatus', 'LEDStatus', 'DetectorStatus', 'ValveStatus', 'Status', 'TapeAdvanceCount', 'TapeAdvanceLeft', 'CPU', 'DiskSpace', 'NumConnections']

COLUMNS_AE33_EXTERNAL_DEVICE_DATA = ['SerialNumber', 'ID', 'DataID', 'DeviceID', 'DeviceData']

COLUMNS_AE33_SETUP = ['serial', 'ID', 'SerialNumber', 'Timestamp', 'FirmwareVer', 'SoftwareVer', 'DataCenterIP', 'AutoConnect', 'InletFilter', 'Timebase', 'SG1', 'SG2', 'SG3', 'SG4', 'SG5', 'SG6', 'SG7', 'C', 'Area', 'Zeta', 'Aff', 'Abb', 'Pressure', 'Temp', 'ATNf1', 'ATNf2', 'Kmax', 'Kmin', 'Flow', 'FlowRepStd', 'PumpPresetValue', 'FlowFormulaA0', 'FlowFormulaA1', 'FlowFormulaB0', 'FlowFormulaB1', 'FlowFormulaC0', 'FlowFormulaC1', 'FlowFormulaD', 'FlowFormulaE', 'FlowFormulaF', 'TAtype', 'TAatnMax', 'TAinterval', 'TAtime', 'TapeRightFormulaK', 'TapeRightFormulaN', 'TapeLeftFormulaK', 'TapeLeftFormulaN', 'WarmUpInterval', 'AutoTestEnabled', 'AutoTestType', 'AutoTestDay', 'AutoTestTime', 'MeasureTimeStamp', 'HomeInfo', 'Display', 'About', 'DST', 'TimeZone', 'TapeAdvanceAdjust', 'ExternalID', 'BHparamID', 'TimeSync', 'DHCP', 'InstrumentIP', 'SubnetMask', 'Gateway', 'Baud', 'NTPserver']

In [4]:
def receive_all(sock, buffer_size=4096, timeout=2.0):
    sock.settimeout(timeout)
    chunks = []
    while True:
        try:
            chunk = sock.recv(buffer_size)
            if not chunk:
                break
            chunks.append(chunk)
        except socket.timeout:
            break
    return b''.join(chunks)

def request_dataview(ip, port, command, buffer_size=4096, timeout=2.0):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(timeout)
        s.connect((ip, port))
        s.send(command.encode())
        return receive_all(s, buffer_size=buffer_size, timeout=timeout)

In [5]:
### command HELLO
command_ae33 = 'HELLO\r\n'

data = request_dataview(ip=INSTRUMENT_IP,
                        port=INSTRUMENT_PORT,
                        command=command_ae33,
                        buffer_size=BUFFER_SIZE,
                        timeout=RECV_TIMEOUT)
text = data.decode(errors='replace')
print('Received', len(data), 'bytes')
print(text)

Received 421 bytes
AE33>Aethalometer information
Server type: AE33
Instrument serialnumber: AE33-S06-00565
Time: 09/Jun/2026 06:09:52
Database version: 1.7.3
Connected Clients: 3
ID: 1225574  IP: 10.10.10.253:49752  Time: 26 May 2026 15:47:10  Streaming: True
ID: 1225572  IP: 10.10.10.253:49741  Time: 26 May 2026 15:47:07  Streaming: True
ID: 1333619  IP: 192.168.45.37:59178  Time: 09 Jun 2026 06:09:52  Streaming: False

AE33>


In [6]:
### command MAXID Data
command_ae33 = 'MAXID Data\r\n'

data = request_dataview(ip=INSTRUMENT_IP,
                        port=INSTRUMENT_PORT,
                        command=command_ae33,
                        buffer_size=BUFFER_SIZE,
                        timeout=RECV_TIMEOUT)
text = data.decode(errors='replace')
print('Received', len(data), 'bytes')
print(text)

# # parse data
max_data_id = int(re.search(r'AE33>(\d+)', text).group(1))
print()
print(max_data_id)


Received 19 bytes
AE33>3156546
AE33>

3156546


In [7]:
### command FETCH Data - collect last 5 rows from table Data
command_ae33 = f'FETCH Data {max_data_id-5} {max_data_id}\r\n'

data = request_dataview(ip=INSTRUMENT_IP,
                        port=INSTRUMENT_PORT,
                        command=command_ae33,
                        buffer_size=BUFFER_SIZE,
                        timeout=RECV_TIMEOUT)
text = data.decode(errors='replace')
print('Received', len(data), 'bytes')
print(text)

# in the end, you can parse data, for example convert it into pandas dataframe
df_ae33_data = pd.read_csv(StringIO(text.replace('AE33>', '').strip()),
                           sep='|',
                           names=COLUMNS_AE33_DATA)
print()
print(df_ae33_data)

Received 2827 bytes
AE33>AE33>AE33-S06-00565|3156541|6/9/2026 6:03:00 AM|6/9/2026 6:04:00 AM|26|7/29/2025 7:24:13 AM|935731|638056|656582|898870|655434|692480|914668|680044|687682|916042|677873|681399|912951|791858|808497|804910|912873|928840|873491|917623|916982|46|-60|48|76|32|79|65|17|68|37|-61|39|76|14|78|15|-84|16|18|-154|18|0.006439992|0.006718565|0.007553311|0.008686225|0.009296549|0.01159205|0.01154316|100|101325.0|21.1|3587|1269|4856|34.0|39.0|35.0|0|10|10|0|0|6061|290|8|5243|9
AE33>AE33-S06-00565|3156542|6/9/2026 6:04:00 AM|6/9/2026 6:05:00 AM|26|7/29/2025 7:24:13 AM|935746|637994|656568|898877|655397|692482|914688|680024|687700|916141|677905|681475|913064|791932|808604|804946|912884|928881|873589|917701|917086|97|91|104|70|15|73|62|-15|64|84|-13|86|47|-37|48|67|3|68|60|-8|61|0.006439992|0.006718565|0.007553311|0.008686225|0.009296549|0.01159205|0.01154316|19.1|101325.0|21.1|3587|1269|4856|34.0|40.0|35.0|0|10|10|0|0|6061|290|6|5243|4
AE33>AE33-S06-00565|3156543|6/9/2026 6:05:

In [8]:
### collect data from Setup table
command_ae33 = f'FETCH SETUP\r\n'

data = request_dataview(ip=INSTRUMENT_IP,
                        port=INSTRUMENT_PORT,
                        command=command_ae33,
                        buffer_size=BUFFER_SIZE,
                        timeout=RECV_TIMEOUT)
text = data.decode(errors='replace')
print('Received', len(data), 'bytes')
print(text)

# in the end, you can parse data, for example convert it into pandas dataframe
df_ae33_setup = pd.read_csv(StringIO(text.replace('AE33>', '').strip()),
                            sep='|',
                            names=COLUMNS_AE33_SETUP)
# drop duplicated serial column from setup data
df_ae33_setup = df_ae33_setup.drop(columns=['serial'], errors='ignore')
print()
print(df_ae33_setup)

Received 6028 bytes
AE33>AE33-S06-00565|1|AE33-S06-00565|10/30/2020 4:47:54 PM|533|1.4.9.1|10.10.10.253:8007|1|0|60|18.47|14.54|13.14|11.58|10.35|7.77|7.19|1.39|0.785|0.02|1|2|101325|25.00|10|30|0.015|-0.005|2000|1|585|-1952.80444335937|-2505.5458984375|11.6349792480469|12.9688386917114|0.000753700849600136|-0.000350014335708693|166.16487121582|0.0840134471654892|-4.06122467211389E-07|1|120|12|3/13/2017 10:57:40 AM|1.04204201698303|1.88889074325562|1.12658226490021|-45.7848091125488|1|1|0|2|1/1/2014 12:00:00 AM|1|0|1|0|0|Coordinated Universal Time|10|1|1|1|1|192.168.0.2|255.255.255.0|192.168.0.1|115200|pool.ntp.org
AE33>AE33>AE33-S06-00565|13|AE33-S06-00565|9/14/2021 7:23:04 AM|540|1.5.0.1|10.10.10.253:8007|1|0|60|18.47|14.54|13.14|11.58|10.35|7.77|7.19|1.39|0.785|0.02|1|2|101325|25.00|10|30|0.015|-0.005|2000|1|585|-1952.80444335937|-2505.5458984375|11.6349792480469|12.9688386917114|0.000753700849600136|-0.000350014335708693|166.16487121582|0.0840134471654892|-4.06122467211389E-07|1|12

In [9]:
### collect data from ExtDeviceData table - get MAXID and then fetch last 5 rows

command_ae33 = 'MAXID ExtDeviceData\r\n'
data = request_dataview(ip=INSTRUMENT_IP,
                        port=INSTRUMENT_PORT,
                        command=command_ae33,
                        buffer_size=BUFFER_SIZE,
                        timeout=RECV_TIMEOUT)
text = data.decode(errors='replace')
max_ext_device_data_id = int(re.search(r'AE33>(\d+)', text).group(1))

command_ae33 = f'FETCH ExtDeviceData {max_ext_device_data_id-5} {max_ext_device_data_id}\r\n'
data = request_dataview(ip=INSTRUMENT_IP,
                        port=INSTRUMENT_PORT,
                        command=command_ae33,
                        buffer_size=BUFFER_SIZE,
                        timeout=RECV_TIMEOUT)
text = data.decode(errors='replace')
print('Received', len(data), 'bytes')
print(text)

# in the end, you can parse data, for example convert it into pandas dataframe
df_ae33_device_data = pd.read_csv(StringIO(text.replace('AE33>', '').strip()),
                           sep='|',
                           names=COLUMNS_AE33_EXTERNAL_DEVICE_DATA)
print()
print(df_ae33_device_data)

Received 268 bytes
AE33>AE33-S06-00565|4439982|3156542|15|97
AE33>AE33>AE33-S06-00565|4439983|3156543|15|97
AE33>AE33-S06-00565|4439984|3156544|15|97
AE33>AE33-S06-00565|4439985|3156545|15|97
AE33>AE33-S06-00565|4439986|3156546|15|97
AE33>AE33-S06-00565|4439987|3156547|15|97
AE33>

     SerialNumber       ID   DataID  DeviceID  DeviceData
0  AE33-S06-00565  4439982  3156542        15          97
1  AE33-S06-00565  4439983  3156543        15          97
2  AE33-S06-00565  4439984  3156544        15          97
3  AE33-S06-00565  4439985  3156545        15          97
4  AE33-S06-00565  4439986  3156546        15          97
5  AE33-S06-00565  4439987  3156547        15          97
